# Day 2: HuggingFace Pipelines

> This notebook documents **Day 2 only**.  
> Later days have separate notebooks.
> Colab setup, GPU verification, and HuggingFace authentication were introduced in Day 1 and are reused here without re-explanation.


## Overview

The HuggingFace transformers library provides APIs at two different levels. The **High Level API** for using open-source models for typical inference tasks is called "pipelines". It's incredibly easy to use.

## Learning Objectives

- Understand High-Level Pipeline API
- Learn Training vs Inference distinction
- Explore multiple pipeline tasks
- Understand model selection in pipelines
- Learn Colab-specific pro-tips

## Resources

- [HuggingFace Pipelines Colab](https://colab.research.google.com/drive/1aMaEw8A56xs0bRM4lu8z7ou18jqyybGm?usp=sharing)
- [HuggingFace Transformers Docs](https://huggingface.co/docs/transformers)


## Colab Pro-Tips

### Pro-Tip 1: Warnings
Data Science code often gives warnings and messages. They can mostly be safely ignored! Glance over them, and if something goes wrong later, perhaps they can give you a clue.

### Pro-Tip 2: Misleading CUDA Errors
In the middle of running a Colab, you might get an error like:
```
Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]
```

**This is a super-misleading error message!** Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. `Kernel menu → Disconnect and delete runtime`
2. Reload the colab from fresh and `Edit menu → Clear All Outputs`
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

---

## Training vs Inference

You may already know this, but just in case you're not familiar with the word "inference":

When working with Data Science models, you could be carrying out 2 very different activities: **training** and **inference**.

### 1. Training
Training is when you provide a model with data for it to adapt to get better at a task in the future. It does this by updating its internal settings - the parameters or weights of the model. If you're Training a model that's already had some training, the activity is called "fine-tuning".

### 2. Inference
Inference is when you are working with a model that has already been trained. You are using that model to produce new outputs on new inputs, taking advantage of everything it learned while it was being trained. Inference is also sometimes referred to as "Execution" or "Running a model".

**Key Points:**
- All of our use of APIs for GPT, Claude and Gemini in the last weeks are examples of inference
- The "P" in GPT stands for "Pre-trained", meaning that it has already been trained with data (lots of it!)
- The pipelines API in HuggingFace is **only for use for inference** - running a model that has already been trained
- In week 7 we will be training our own model, and we will need to use the more advanced HuggingFace APIs

---

## Setup

### 1. Install Dependencies


In [ ]:
# Pip installs should come at the top line.
# If your Kernel ever resets, you need to run this again.

!pip install -q --upgrade datasets==3.6.0


### 2. Verify GPU

> Always run this cell immediately after connecting to a runtime.


In [ ]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")


### 3. Imports


In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio, display


### 4. Set up HuggingFace Token

**Important Note - Hugging Face account**

In Day 1, we set up a FREE account on https://huggingface.co

If you skipped this:
Please go back and do it! Then go to the Avatar menu, Tokens, and create an API token. And make sure it has WRITE permissions! And then add it to the secrets on the left by pressing the key button.

If you did this (thank you!)
Click on the Key button and turn on the switch so that this notebook gets access to your Hugging Face key.


In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)


## Using Pipelines from Hugging Face

A simple way to run inference for common tasks, without worrying about all the plumbing, picking reasonable defaults.

**How it works:**

**STEP 1:** Create a pipeline - a function you can then call
```python
my_pipeline = pipeline(task, model=xx, device=xx)
```
If you don't specify a model, then Hugging Face picks one for you that's the default for the task. Specify "cuda" for the device to use an NVIDIA GPU like the one on the T4. Specify "mps" on a Mac.

**STEP 2:** Then call it as many times as you want:
```python
my_pipeline(input1)
my_pipeline(input2)
```


## Pipeline Tasks

### 1. Sentiment Analysis


In [ ]:
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device="cuda")
result = my_simple_sentiment_analyzer("I'm super excited to be on the way to LLM mastery!")
print(result)

result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

# Using a better model (multilingual)
better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)


### 2. Named Entity Recognition (NER)


In [ ]:
ner = pipeline("ner", device="cuda")
result = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entity in result:
  print(entity)


### 3. Question Answering with Context


In [ ]:
question = "What are Hugging Face pipelines?"
context = "Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device="cuda")
result = question_answerer(question=question, context=context)
print(result)


### 4. Text Summarization


In [ ]:
summarizer = pipeline("summarization", device="cuda")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""
summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])


### 5. Translation


In [ ]:
# English to French (default model)
translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

# English to Spanish (specifying model)
# All translation models: https://huggingface.co/models?pipeline_tag=translation&sort=trending
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])


### 6. Zero-shot Classification


In [ ]:
classifier = pipeline("zero-shot-classification", device="cuda")
result = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(result)


### 7. Text Generation


In [ ]:
generator = pipeline("text-generation", device="cuda")
result = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(result[0]['generated_text'])


### 8. Image Generation (Diffusers)

**Note:** Pipelines can be used for diffusion models as well as transformers. This was also shown in Day 1, but now explained as part of the pipelines ecosystem.


In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)


### 9. Audio Generation (Text-to-Speech)

**Note:** This was also shown in Day 1, but now explained as part of the pipelines ecosystem.


In [ ]:
synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])


## Key Learnings

### High-Level Pipeline API
- Pipelines are a high-level API for inference only (not training)
- Two-step pattern: Create pipeline → Call pipeline
- Default models automatically selected if not specified
- GPU acceleration via `device="cuda"` parameter
- Same simple API pattern works across all tasks

### Training vs Inference
- **Training:** Model learns from data, updates parameters
- **Inference:** Using trained model to produce outputs
- Pipelines API is only for inference
- Week 7 will cover training (need lower-level APIs)

### Pipeline Tasks
- Sentiment Analysis, NER, Q&A, Summarization, Translation
- Zero-shot Classification, Text Generation
- Image Generation (Diffusers), Audio Generation (TTS)
- Same API pattern: `pipeline(task, model=optional, device="cuda")`

### Model Selection
- Default models are good starting points
- Can specify custom models for better results
- Different models have different strengths
- Pattern: Start with default → Try specific models if needed

### Colab Pro-Tips
- Warnings can mostly be ignored (glance, don't worry)
- CUDA errors often mean runtime switch (not package issue)
- Solution: Full reset → Verify GPU → Run from top
